In [1]:
%matplotlib inline
import matplotlib.pyplot as plt

In [2]:
# Setup

%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import shap
from mlflow.tracking import MlflowClient
from sklearn.base import clone
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay
from sklearn.model_selection import cross_val_predict

from churnguard.evaluation import THEORETICAL_THRESHOLD, metrics_at_threshold
from churnguard.features import split_X_y
from churnguard.split import TRAIN_PATH
from churnguard.tracking import EXPERIMENT_NAME, MODEL_NAME, setup_mlflow
from churnguard.train import CV

setup_mlflow()
X, y = split_X_y(pd.read_parquet(TRAIN_PATH))

In [3]:
# Compare all runs as a table

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.stage = 'cv_comparison'",
)
cols = ["tags.mlflow.runName", "metrics.cv_pr_auc_mean", "metrics.cv_pr_auc_std",
        "metrics.cv_roc_auc_mean", "metrics.cv_recall_mean",
        "metrics.cv_precision_mean", "metrics.cv_brier_mean"]
runs[cols].sort_values("metrics.cv_pr_auc_mean", ascending=False).round(3)

,tags.mlflow.runName,metrics.cv_pr_auc_mean,metrics.cv_pr_auc_std,metrics.cv_roc_auc_mean,metrics.cv_recall_mean,metrics.cv_precision_mean,metrics.cv_brier_mean
0,xgboost_tuned,0.665,0.019,0.848,0.799,0.524,0.160
4,logreg_unweighted,0.659,0.019,0.846,0.547,0.654,0.135
3,logreg_balanced,0.658,0.019,0.846,0.802,0.518,0.165
1,xgboost,0.655,0.018,0.841,0.760,0.532,0.158
2,random_forest,0.654,0.021,0.843,0.714,0.565,0.149
5,baseline_dummy,0.265,0.000,0.500,0.000,0.000,0.195


In [4]:
# Load the candidate from the registry

client = MlflowClient()
mv = client.get_model_version_by_alias(MODEL_NAME, "candidate")
threshold = float(mv.tags["threshold"])
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@candidate")

print(f"Version {mv.version} | type {mv.tags['model_type']} | threshold {threshold}")
model

Version 1 | type logreg_unweighted | threshold 0.32


Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('engineer',
                                  FunctionTransformer(func=<function engineer_features at 0x000001A65EA30FE0>)),
                                 ('columns',
                                  ColumnTransformer(transformers=[('num',
                                                                   Pipeline(steps=[('impute',
                                                                                    SimpleImputer(strategy='median')),
                                                                                   ('scale',
                                                                                    StandardScaler())]),
                                                                   ['tenure',
                                                                    'monthly_charges',
                                                                    'total_charges',
                                                                    'num_addon_services',
                                                                    'charge_diff'...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False),
                                                                   ['partner',
                                                                    'dependents',
                                                                    'phone_service',
                                                                    'multiple_lines',
                                                                    'internet_service',
                                                                    'online_security',
                                                                    'online_backup',
                                                                    'device_protection',
                                                                    'tech_support',
                                                                    'streaming_tv',
                                                                    'streaming_movies',
                                                                    'paperless_billing',
                                                                    'payment_method'])],
                                                    verbose_feature_names_out=False))])),
                ('model', LogisticRegression(max_iter=1000))])

In [5]:
# Out-of-fold evaluation plots

oof = cross_val_predict(clone(model), X, y, cv=CV, method="predict_proba")[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay.from_predictions(y, (oof >= threshold).astype(int), ax=axes[0])
axes[0].set_title(f"Confusion matrix @ {threshold:.2f}")
PrecisionRecallDisplay.from_predictions(y, oof, ax=axes[1])
axes[1].set_title("Precision-recall curve")
CalibrationDisplay.from_predictions(y, oof, n_bins=10, ax=axes[2])
axes[2].set_title("Calibration")
plt.tight_layout()
plt.show()

print(f"Chosen threshold: {threshold:.2f} | theoretical (if calibrated): {THEORETICAL_THRESHOLD:.2f}")
pd.Series(metrics_at_threshold(y, oof, threshold)).round(3)

Chosen threshold: 0.32 | theoretical (if calibrated): 0.33


C:\Users\Pejman\AppData\Local\Temp\ipykernel_18716\78627561.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


roc_auc             0.845
pr_auc              0.656
brier               0.135
recall              0.750
precision           0.556
f1                  0.638
profit          67250.000
contact_rate        0.358
dtype: float64

In [7]:
# SHAP, global explanation

pre, clf = model[:-1], model[-1]       # preprocessing part, model part
Xt = pre.transform(X)                   # features exactly as the model sees them
sample = Xt.sample(500, random_state=42)

explainer = shap.Explainer(clf, sample)
sv = explainer(sample)
if sv.values.ndim == 3:                 # some models return values for both classes
    sv = sv[..., 1]

shap.plots.beeswarm(sv, max_display=15)

Background dataset has 500 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=500 when initializing the masker.
e:\My Courses\LLM_LLAMA3_ART_Projects\AI_Engineer_Projects_2026\churnguard\churn-envy\Lib\site-packages\shap\plots\_beeswarm.py:503: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# SHAP, one customer (local explanation)

risk = clf.predict_proba(sample)[:, 1]
idx = int(np.argmax(risk))
print(f"Highest-risk customer in sample: p(churn) = {risk[idx]:.2f}")
shap.plots.waterfall(sv[idx], max_display=10)

Highest-risk customer in sample: p(churn) = 0.85


e:\My Courses\LLM_LLAMA3_ART_Projects\AI_Engineer_Projects_2026\churnguard\churn-envy\Lib\site-packages\shap\plots\_waterfall.py:371: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
